# Sistemas de Recomendacion - Tu Futuro segun la Data

Notebook de exploracion del Adult Income Dataset. Se documenta la carga, limpieza, modelado supervisado y sistema de recomendacion interpretativo.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

import joblib
import pandas as pd

from app import MODEL_PATH, METRICS_PATH, RECOMMENDATIONS_PATH, train_model_and_recommender
from utils import TARGET, load_adult_income_dataset, simulated_profiles

## 1. Carga del conjunto de datos

In [2]:
df = load_adult_income_dataset(PROJECT_ROOT / "data" / "raw" / "adult-census-income.csv")
print(df.shape)
df.head()

(32537, 16)


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income,income_high
0,90,Unknown,77053,HS-grad,9,Widowed,Unknown,Not-in-family,White,Female,0,4356,40,United-States,<=50K,False
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K,False
2,66,Unknown,186061,Some-college,10,Widowed,Unknown,Unmarried,Black,Female,0,4356,40,United-States,<=50K,False
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K,False
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K,False


In [3]:
df[TARGET].value_counts(normalize=True).rename("proportion").to_frame().join(
    df[TARGET].value_counts().rename("count")
)

,proportion,count
income_high,,
False,0.759074,24698
True,0.240926,7839


## 2. Revision de variables

El dataset combina variables numericas y categoricas. Los valores `?` fueron convertidos a `Unknown` en las variables categoricas afectadas.

In [4]:
df.dtypes.to_frame("dtype")

,dtype
age,int64
workclass,str
fnlwgt,int64
education,str
education_num,int64
marital_status,str
occupation,str
relationship,str
race,str
sex,str


In [5]:
categorical_summary = {
    column: df[column].value_counts().head(5).to_dict()
    for column in df.select_dtypes(include="object").columns
    if column != "income"
}
categorical_summary

C:\Users\frang\AppData\Local\Temp\ipykernel_18912\1897245520.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in df.select_dtypes(include="object").columns


{'workclass': {'Private': 22673,
  'Self-emp-not-inc': 2540,
  'Local-gov': 2093,
  'Unknown': 1836,
  'State-gov': 1298},
 'education': {'HS-grad': 10494,
  'Some-college': 7282,
  'Bachelors': 5353,
  'Masters': 1722,
  'Assoc-voc': 1382},
 'marital_status': {'Married-civ-spouse': 14970,
  'Never-married': 10667,
  'Divorced': 4441,
  'Separated': 1025,
  'Widowed': 993},
 'occupation': {'Prof-specialty': 4136,
  'Craft-repair': 4094,
  'Exec-managerial': 4065,
  'Adm-clerical': 3768,
  'Sales': 3650},
 'relationship': {'Husband': 13187,
  'Not-in-family': 8292,
  'Own-child': 5064,
  'Unmarried': 3445,
  'Wife': 1568},
 'race': {'White': 27795,
  'Black': 3122,
  'Asian-Pac-Islander': 1038,
  'Amer-Indian-Eskimo': 311,
  'Other': 271},
 'sex': {'Male': 21775, 'Female': 10762},
 'native_country': {'United-States': 29153,
  'Mexico': 639,
  'Unknown': 582,
  'Philippines': 198,
  'Germany': 137}}

## 3. Definicion del recomendador

- Usuario: persona adulta representada por su perfil socioeconomico.
- Recomendacion: trayectorias accionables de educacion, ocupacion, tipo de trabajo y horas semanales.
- Enfoque: hibrido, combinando probabilidad supervisada de `>50K` y vecinos similares de alto ingreso.

## 4. Entrenamiento, optimizacion y artefactos

In [6]:
results = train_model_and_recommender()
results["optimized_classifier"]["best_params"]

Fitting 5 folds for each of 8 candidates, totalling 40 fits


{'classifier__C': 0.3, 'classifier__class_weight': 'balanced'}

In [7]:
pd.DataFrame([
    {"model": "baseline", **{k: results["baseline_classifier"][k] for k in ["accuracy", "precision_high_income", "recall_high_income", "f1_high_income", "roc_auc"]}},
    {"model": "optimized", **{k: results["optimized_classifier"][k] for k in ["accuracy", "precision_high_income", "recall_high_income", "f1_high_income", "roc_auc"]}},
])

,model,accuracy,precision_high_income,recall_high_income,f1_high_income,roc_auc
0,baseline,0.851721,0.736471,0.598852,0.660570,0.902001
1,optimized,0.808543,0.570738,0.828444,0.675858,0.902258


## 5. Pruebas con perfiles simulados

In [8]:
artifact = joblib.load(MODEL_PATH)
recommender = artifact["recommender"]
profile = simulated_profiles()["young_part_time_hs"]
recommendation = recommender.recommend(profile)
print("Probabilidad base:", recommendation.base_probability)
pd.DataFrame(recommendation.recommendations)

Probabilidad base: 0.0175


,recommendation,estimated_probability,probability_uplift,changes,rationale
0,Avanzar a Doctorate,0.1428,0.1253,"{'education_num': 16, 'education': 'Doctorate'}",La educacion suele aumentar el acceso a ocupac...
1,Avanzar a Prof-school,0.1190,0.1015,"{'education_num': 15, 'education': 'Prof-school'}",La educacion suele aumentar el acceso a ocupac...
2,Avanzar a Masters,0.0699,0.0524,"{'education_num': 14, 'education': 'Masters'}",La educacion suele aumentar el acceso a ocupac...
3,Avanzar a Bachelors,0.0529,0.0354,"{'education_num': 13, 'education': 'Bachelors'}",La educacion suele aumentar el acceso a ocupac...
4,Aumentar disponibilidad a 50 horas semanales,0.0402,0.0226,{'hours_per_week': 50},El modelo evalua si una mayor disponibilidad l...


In [9]:
pd.DataFrame(recommendation.similar_successful_profiles)

,similarity,education,occupation,workclass,hours_per_week
0,0.7897,HS-grad,Adm-clerical,Private,20
1,0.7678,HS-grad,Adm-clerical,Private,40
2,0.7357,HS-grad,Adm-clerical,Private,40
3,0.7238,HS-grad,Adm-clerical,Private,25
4,0.7205,HS-grad,Adm-clerical,Private,35


In [10]:
print(MODEL_PATH.exists(), METRICS_PATH.exists(), RECOMMENDATIONS_PATH.exists())
print(MODEL_PATH.relative_to(PROJECT_ROOT))
print(METRICS_PATH.relative_to(PROJECT_ROOT))
print(RECOMMENDATIONS_PATH.relative_to(PROJECT_ROOT))

True True True
models\adult_income_recommender.joblib
models\adult_income_metrics.json
models\sample_recommendations.json
